In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_2_CVAE import CVAE
from e_1_run_cvae import train_chunk
#from e_1_run_cvae_time_check import train_chunk
#from e_2_CVAE_norm import CVAE as CVAE_norm
#from e_1_run_cvae_norm import train_chunk_norm

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # hes or bs
barr_type = 'van' # van or barr
opt_type = 'call' # call or put
chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# training

In [27]:
# CVAE training settings
dim_z       = 8 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 92 # 1 chunk train : 3m, 92+9:283m / 5+5:40m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 10
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk383.pt"

In [28]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_8_128_4096_None_0.0001_1_[15, 24, 78]_chunk291.pt | 완료 chunks=291
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=291->383 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=10 | bn_chunks=None | warmup_chunks=None
Chunk step   292 | epoch    4 chunk   1/97 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -5.1393 | KL: 4.4069 | Total: -0.7324
Chunk step   293 | epoch    4 chunk   2/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.1316 | KL: 4.4152 | Total: -0.7164
Chunk step   294 | epoch    4 chunk   3/97 | file_idx  12 | BN off    | beta_eff: 1.0000 | Recon: -5.1394 | KL: 4.4063 | Total: -0.7330
Chunk step   295 | epoch    4 chunk   4/97 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -5.1475 | KL: 4.4178 | Total: -0.7297
Chunk step   296 | epoch    4 chunk   5/97 | file_idx  94 | BN off    | beta_eff: 1.0000 | Recon: -5.1490 | KL: 4.4189 | Total: -0.7301
Chunk step   297 | epoch 

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk383.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388.pt" 


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_8_128_4096_None_0.0001_1_[15, 24, 78]_chunk383.pt | 완료 chunks=383
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=383->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None


Chunk step   384 | epoch    4 chunk  93/97 | file_idx   4 | BN off    | beta_eff: 1.0000 | Recon: -5.2539 | KL: 4.5172 | Total: -0.7367
Validation @ chunk   384 | Recon: -5.2722 | KL: 4.5290 | Total: -0.7432 | KL_dim: [3e-06, 2e-06, 6e-06, 2e-06, 1.470462, 2e-06, 3.058508, 3e-06]
Chunk step   385 | epoch    4 chunk  94/97 | file_idx  69 | BN off    | beta_eff: 1.0000 | Recon: -5.2453 | KL: 4.5204 | Total: -0.7249
Validation @ chunk   385 | Recon: -5.2431 | KL: 4.5035 | Total: -0.7396 | KL_dim: [2e-06, 2e-06, 2e-06, 2e-06, 1.457665, 2e-06, 3.045816, 2e-06]
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.2574 | KL: 4.5116 | Total: -0.7458
Validation @ chunk   386 | Recon: -5.2377 | KL: 4.5070 | Total: -0.7307 | KL_dim: [4e-06, 2e-06, 3e-06, 8.7e-05, 1.453884, 2e-06, 3.053014, 4e-06]
Chunk step   387 | epoch    4 chunk  96/97 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -5.2531 | KL: 4.5099 | Total: -0.7432
Validation @ chunk 

# BN = 5

In [ ]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = 5 # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-5
l2          = 3
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 92
validation_chunk_idxs = [15,24,78]
val_every_chunks = 10
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk383.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_0.0001_1_[15, 24, 78]_chunk291.pt | 완료 chunks=291
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=291->383 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=10 | bn_chunks=5 | warmup_chunks=None
Chunk step   292 | epoch    4 chunk   1/97 | file_idx  52 | BN frozen | beta_eff: 1.0000 | Recon: -5.2013 | KL: 4.4660 | Total: -0.7353
Chunk step   293 | epoch    4 chunk   2/97 | file_idx   7 | BN frozen | beta_eff: 1.0000 | Recon: -5.1944 | KL: 4.4750 | Total: -0.7194
Chunk step   294 | epoch    4 chunk   3/97 | file_idx  12 | BN frozen | beta_eff: 1.0000 | Recon: -5.1999 | KL: 4.4639 | Total: -0.7360
Chunk step   295 | epoch    4 chunk   4/97 | file_idx  37 | BN frozen | beta_eff: 1.0000 | Recon: -5.2035 | KL: 4.4711 | Total: -0.7324
Chunk step   296 | epoch    4 chunk   5/97 | file_idx  94 | BN frozen | beta_eff: 1.0000 | Recon: -5.2012 | KL: 4.4687 | Total: -0.7325
Chunk step   297 | epoch    4 c

In [3]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk383.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk388.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_3-0.0001_4-1e-06_1_[15, 24, 78]_chunk383.pt | 완료 chunks=383
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=383->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=5 | warmup_chunks=None
Chunk step   384 | epoch    4 chunk  93/97 | file_idx   4 | BN frozen | beta_eff: 1.0000 | Recon: -5.2304 | KL: 4.4934 | Total: -0.7370
Validation @ chunk   384 | Recon: -5.2282 | KL: 4.4857 | Total: -0.7425 | KL_dim: [1.436707, 3.048991]
Chunk step   385 | epoch    4 chunk  94/97 | file_idx  69 | BN frozen | beta_eff: 1.0000 | Recon: -5.2268 | KL: 4.5010 | Total: -0.7258
Validation @ chunk   385 | Recon: -5.2273 | KL: 4.4858 | Total: -0.7415 | KL_dim: [1.42515, 3.060651]
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN frozen | beta_eff: 1.0000 | Recon: -5.2385 | KL: 4.4922 | Total: -0.7463
Validation @ chunk   386 | Recon: -5.2095 | KL: 4.4750 | Total: -0.7345 | KL_dim: [1.42636

In [8]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = 5 # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 9
lr2         = 1e-5
l2          = 10
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 92
validation_chunk_idxs = [15,24,78]
val_every_chunks = 10
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk873.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk965.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_0.0001_1_[15, 24, 78]_chunk873.pt | 완료 chunks=873
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=873->965 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=10 | bn_chunks=5 | warmup_chunks=None
Chunk step   874 | epoch   10 chunk   1/97 | file_idx  50 | BN frozen | beta_eff: 1.0000 | Recon: -5.3109 | KL: 4.5648 | Total: -0.7460
Chunk step   875 | epoch   10 chunk   2/97 | file_idx  44 | BN frozen | beta_eff: 1.0000 | Recon: -5.3068 | KL: 4.5632 | Total: -0.7436
Chunk step   876 | epoch   10 chunk   3/97 | file_idx  89 | BN frozen | beta_eff: 1.0000 | Recon: -5.3056 | KL: 4.5665 | Total: -0.7390
Chunk step   877 | epoch   10 chunk   4/97 | file_idx  61 | BN frozen | beta_eff: 1.0000 | Recon: -5.3048 | KL: 4.5651 | Total: -0.7398
Chunk step   878 | epoch   10 chunk   5/97 | file_idx  11 | BN frozen | beta_eff: 1.0000 | Recon: -5.3142 | KL: 4.5609 | Total: -0.7533
Chunk step   879 | epoch   10 c

KeyboardInterrupt: 

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk965.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk970.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

# X,M norm

In [ ]:
# CVAE training settings
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 0.001 # 0.001,0.0003, 0.0005
beta        = 0.9
warmup_chunks = None # None or num
num_chunks  = 70
validation_chunk_idxs = [15,24,78]
val_every_chunks = 3
resume_path = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk30.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk100.pt"

In [6]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_norm(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    x_mean=bs_stats["x_mean"],
    x_std=bs_stats["x_std"],
    m_mean=bs_stats["m_mean"],
    m_std=bs_stats["m_std"],
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae_xm_norm/bs/cvae_bs_8_128_4096_None_0.001_0.9_None_chunk30.pt | 완료 chunks=30
학습 시작 | 이번 실행 chunks=70 | 진행 chunks=30->100 | files/epoch=100 | bn_chunks=None | warmup_chunks=None
Chunk step    31 | epoch    1 chunk  31/100 | file_idx  30 | BN off    | beta_eff: 0.9000 | Recon: -4.9477 | KL: 5.3827 | Total: -0.1033
Chunk step    32 | epoch    1 chunk  32/100 | file_idx  29 | BN off    | beta_eff: 0.9000 | Recon: -4.8912 | KL: 6.1971 | Total: 0.6862
Chunk step    33 | epoch    1 chunk  33/100 | file_idx  79 | BN off    | beta_eff: 0.9000 | Recon: -4.9538 | KL: 6.2571 | Total: 0.6776
Chunk step    34 | epoch    1 chunk  34/100 | file_idx  44 | BN off    | beta_eff: 0.9000 | Recon: -5.0059 | KL: 5.4327 | Total: -0.1164
Chunk step    35 | epoch    1 chunk  35/100 | file_idx  71 | BN off    | beta_eff: 0.9000 | Recon: -4.9941 | KL: 5.4282 | Total: -0.1088
Chunk step    36 | epoch    1 chunk  36/100 | file_idx  66 | BN off    | beta_eff: 0.9000 | Rec